# Grid Search de modelos de sueño

Búsqueda de hiperparámetros basada en `model_jhoan_saavedra.ipynb`. Compara Random Forest y LightGBM usando el pipeline compartido `sleep-staging`.

La validación final queda aislada. `GridSearchCV` utiliza únicamente training y aplica `GroupKFold` por sujeto para evitar que las dos noches de una persona aparezcan en folds diferentes.

## 1. Configuración

Los grids incluidos son moderados: 16 combinaciones por modelo y 3 folds internos. Para el informe final puede aumentarse `CV_SPLITS` a 5. El costo total aproximado es `combinaciones × folds` entrenamientos por modelo.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (path / 'data.dvc').is_file())
PACKAGE_SRC = REPO_ROOT / 'packages' / 'sleep-staging' / 'src'
if str(PACKAGE_SRC) not in sys.path:
    sys.path.insert(0, str(PACKAGE_SRC))

from IPython.display import display
from sleep_staging import PreprocessingConfig, PreprocessingPipeline
from sleep_staging.datasets import discover_sleep_telemetry_records
from sleep_staging.training import (
    MlflowConfig, build_supervised_dataset, compare_evaluations,
    create_lightgbm_classifier, create_random_forest_classifier,
    display_evaluation, grouped_grid_search, split_by_subject,
)

DATA_DIR = REPO_ROOT / 'data' / 'sleep-telemetry'
SEED = 42
VALIDATION_SIZE = 0.20
CV_SPLITS = 3
SEARCH_N_JOBS = -1
MAX_RECORDS = None  # usar 6 para una prueba rápida; None usa los 44 registros
MLFLOW = MlflowConfig(
    enabled=False,
    experiment_name='sleep_staging_grid_search',
    tracking_uri='sqlite:///mlflow.db',
)
PIPELINE = PreprocessingPipeline(PreprocessingConfig())

## 2. Preprocesamiento y dataset

Se utiliza exactamente el mismo `PreprocessingPipeline` del notebook de Jhoan y del futuro backend.

In [ ]:
records = discover_sleep_telemetry_records(DATA_DIR)
records = records if MAX_RECORDS is None else records[:MAX_RECORDS]
dataset = build_supervised_dataset(records, PIPELINE)
print(f'Dataset: {dataset.features.shape[0]} épocas x {dataset.features.shape[1]} features')
print(f'Sujetos: {dataset.metadata.subject_id.nunique()} | Registros: {dataset.metadata.record_id.nunique()}')
display(dataset.labels.value_counts().rename('épocas').sort_index().to_frame())

## 3. Holdout de validación por sujeto

Este conjunto se utiliza solamente después de encontrar los mejores hiperparámetros.

In [ ]:
split = split_by_subject(dataset, validation_size=VALIDATION_SIZE, random_state=SEED)
print(f'Training: {split.X_train.shape} | sujetos: {split.train_subjects}')
print(f'Validación: {split.X_validation.shape} | sujetos: {split.validation_subjects}')
assert set(split.train_subjects).isdisjoint(split.validation_subjects)

## 4. Grid Search — Random Forest

Se exploran cantidad de árboles, profundidad, mínimo de muestras por hoja y cantidad de features consideradas en cada división.

In [ ]:
RF_GRID = {
    'n_estimators': [300, 500],
    'max_depth': [None, 20],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 0.5],
}
random_forest = create_random_forest_classifier(random_state=SEED, n_jobs=1)
rf_search = grouped_grid_search(
    random_forest, RF_GRID, split, model_name='Random Forest optimizado',
    n_splits=CV_SPLITS, n_jobs=SEARCH_N_JOBS, verbose=2,
    mlflow_config=MLFLOW,
)
print(f'Mejor F1 macro CV: {rf_search.best_cv_score:.4f}')
print(f'Mejores parámetros: {rf_search.best_parameters}')
display(rf_search.cv_results.head(10))

## 5. Grid Search — LightGBM

Se exploran árboles, tasa de aprendizaje, número de hojas y mínimo de observaciones por hoja.

In [ ]:
LGBM_GRID = {
    'n_estimators': [300, 700],
    'learning_rate': [0.03, 0.08],
    'num_leaves': [31, 63],
    'min_child_samples': [20, 40],
}
lightgbm = create_lightgbm_classifier(random_state=SEED, n_jobs=1)
lgbm_search = grouped_grid_search(
    lightgbm, LGBM_GRID, split, model_name='LightGBM optimizado',
    n_splits=CV_SPLITS, n_jobs=SEARCH_N_JOBS, verbose=2,
    mlflow_config=MLFLOW,
)
print(f'Mejor F1 macro CV: {lgbm_search.best_cv_score:.4f}')
print(f'Mejores parámetros: {lgbm_search.best_parameters}')
display(lgbm_search.cv_results.head(10))

## 6. Mejores modelos y evaluación final

Estas métricas corresponden al holdout de sujetos que no participó en el Grid Search.

In [ ]:
print('MEJOR RANDOM FOREST')
print(rf_search.best_parameters)
print('\nMEJOR LIGHTGBM')
print(lgbm_search.best_parameters)

display(compare_evaluations(
    rf_search.validation_evaluation,
    lgbm_search.validation_evaluation,
).round(4))
display_evaluation(rf_search.validation_evaluation)
display_evaluation(lgbm_search.validation_evaluation)